# NER Dataset Expansion & Deduplication Workflow

This notebook outlines the process of augmenting an existing Named Entity Recognition (NER) dataset while maintaining strict data integrity. Our goal is to expand the **Training Set** with new annotated data without introducing duplicates that might already exist in our current **Train**, **Dev**, or **Test** splits.

## 1. Objective
To merge new annotated paragraphs into our existing pipeline ensuring that:
1. No overlapping content exists between the new data and the existing splits.
2. The final output is a strictly formatted BIO `.txt` file compatible with the **Flair** library.

---

## 2. The Workflow

### Phase I: Data Extraction
We begin by loading our existing datasets. To ensure efficient comparison, we will:
* Extract raw text paragraphs from the current `train.txt`, `dev.txt`, and `test.txt`.
* Load new candidates obtained from langextract with NER from flair comparison.

### Phase II: Conflict Resolution (Deduplication)
Before adding the new data to the training pool, we perform a "leakage check":
* **Filter:** Compare the new candidate paragraphs against the existing sets using a threshold of similarity with fuzzy matching.
* **Action:** If a new paragraph matches an entry in the existing Dev or Test sets, it is **discarded** to preserve the validity of our evaluation metrics.
* **Action:** If a new paragraph matches an entry already in the Train set, it is flagged as a duplicate and ignored to avoid overfitting on redundant samples.

### Phase III: Format Conversion & Export
Once the unique new paragraphs are validated, we reconstruct the dataset:
* **Tokenization:** Ensuring tokens and tags remain aligned.
* **BIO Sequencing:** Writing the data in the standard two-column format:
    > `Word` `Tag`
    
* **Final Assembly:** Merging the validated new data with the original training data into a single `train_expanded.txt`.

In [ ]:
train_set_path = "../../../resources/data/restricted/ner-review/labeled-dataset/train.txt"
dev_set_path = "../../../resources/data/restricted/ner-review/labeled-dataset/dev-review-cb.txt"
test_set_path = "../../../resources/data/restricted/ner-review/labeled-dataset/test-review-cb.txt"

In [ ]:
def return_paragraphs(set_path: str) -> list:
    
    with open(set_path, "r") as f:
        lines = f.readlines()

    paragraphs = []
    current_tokens = []

    for raw_line in lines:
        line = raw_line.rstrip()
        if not line:
            if current_tokens:
                paragraph_text = " ".join(current_tokens).strip()
                paragraphs.append((paragraph_text))
                current_tokens = []
            continue

        try:
            token, label = line.rsplit(" ", 1)
        except ValueError:
            token = line

        current_tokens.append(token)

    if current_tokens:
        paragraph_text = " ".join(current_tokens).strip()
        paragraphs.append((paragraph_text))

    return (paragraphs)

In [ ]:
train_paragraphs = return_paragraphs(train_set_path)
dev_paragraphs = return_paragraphs(dev_set_path)
test_paragraphs = return_paragraphs(test_set_path)

all_existing_text = set(train_paragraphs).union(set(dev_paragraphs)).union(set(test_paragraphs))

In [ ]:
import json

def load_new_candidates(jsonl_path):
    candidates = []
    with open(jsonl_path, 'r', encoding='utf-8') as f:
        for line in f:
            data = json.loads(line)
            text = data.get("text", "")
            entities = data.get("final_entities", [])
            candidates.append({
                "text": text,
                "entities": entities
            })
    return candidates


In [ ]:
train_candidates_path = "../../../resources/data/restricted/ner-langextract-alignment/curated/train_candidates.jsonl"

In [ ]:
train_candidates_data = load_new_candidates(train_candidates_path)

In [ ]:
train_candidates_data[:1]

In [ ]:
from rapidfuzz import fuzz, process, utils
from tqdm import tqdm
import re
from typing import List, Dict, Any, Tuple, Iterable

def perform_internal_deduplication(
    candidates: List[Dict[str, Any]], 
    threshold: float = 95.0
) -> Tuple[List[Dict], List[Dict]]:
    """Identifies and removes duplicate entries within a single list of candidates.

    This function compares each candidate against previously processed unique 
    candidates in the same list. It prioritizes keeping 'labeled' versions 
    over 'unlabeled' versions if a duplicate is found.

    Args:
        candidates: A list of dictionaries containing 'text' and 'entities'.
        threshold: The fuzzy match score (0-100) above which two candidates 
            are considered duplicates. Defaults to 95.0.

    Returns:
        A tuple containing:
            1. unique_candidates: List of distinct samples.
            2. internal_duplicates: List of samples removed as redundant.
    """
    unique_candidates = []
    internal_duplicates = []
    
    unique_texts_processed = []
    
    print(f"Performing internal deduplication on {len(candidates)} samples...")

    for item in tqdm(candidates, desc="Internal Check"):
        raw_text = item.get("text", "").strip()
        if not raw_text:
            continue
            
        processed_text = utils.default_process(raw_text)
        
        match = process.extractOne(
            processed_text, 
            unique_texts_processed, 
            scorer=fuzz.ratio, 
            score_cutoff=threshold
        )
        
        if match:
            # It's a duplicate of something already in unique_candidates
            _, score, index = match
            item['internal_similarity_score'] = score
            item['duplicate_of'] = unique_candidates[index].get('text')
            internal_duplicates.append(item)
        else:
            # It's unique (so far)
            unique_candidates.append(item)
            unique_texts_processed.append(processed_text)

    print("\n" + "="*40)
    print("      INTERNAL DEDUPLICATION SUMMARY")
    print("="*40)
    print(f"  Keep (Unique):      {len(unique_candidates)}")
    print(f"  Remove (Duplicate): {len(internal_duplicates)}")
    print("="*40)
    
    return unique_candidates, internal_duplicates


def is_noise(text: str) -> bool:
    """Checks if a string lacks basic alphanumeric content.

    This acts as a first-pass filter to remove strings consisting only of 
    punctuation, whitespace, or special characters that wouldn't provide 
    meaningful context for NER.

    Args:
        text: The string to be evaluated.

    Returns:
        True if the string contains no letters or numbers, False otherwise.
    """
    if not re.search(r'[a-zA-Z0-9]', text):
        return True
    return False


def perform_deduplication(
    new_candidates: List[Dict[str, Any]], 
    existing_corpus: Iterable[str], 
    threshold: float = 95.0
) -> Tuple[List[Dict], List[Dict], List[Dict], List[Dict], List[Dict]]:
    """Filters new data against an existing corpus to identify duplicates and noise.

    Uses fuzzy string matching (RapidFuzz) to compare new text samples against 
    a known corpus. It categorizes results based on both similarity scores 
    and the presence of entity annotations.

    Args:
        new_candidates: A list of dictionaries, each containing at least a 
            'text' key and optionally an 'entities' or 'final_entities' key.
        existing_corpus: An iterable of strings representing the already 
            known/trained text samples.
        threshold: The fuzzy match score (0-100) above which a sample is 
            considered a duplicate. Defaults to 95.0.

    Returns:
        A tuple containing five lists in the following order:
            1. clean_labeled: New unique samples with entities.
            2. clean_unlabeled: New unique samples without entities.
            3. duplicates_with_labels: Duplicate samples that had entities.
            4. duplicates_no_labels: Duplicate samples without entities.
            5. discarded_noise: Samples rejected for lacking alphanumeric content.
    """

    clean_labeled = []
    clean_unlabeled = []
    duplicates_with_labels = []
    duplicates_no_labels = []
    discarded_noise = []
    
    corpus_as_list = list(existing_corpus)
    # Pre-processing the corpus once to speed up fuzzy matching
    normalized_corpus = [utils.default_process(text) for text in corpus_as_list]
    
    print(f"Analyzing {len(new_candidates)} candidates using RapidFuzz...")
    
    for item in tqdm(new_candidates, desc="Deduplicating"):
        raw_text = item.get("text", "").strip()

        # Step 1: Noise Filtering
        if not raw_text or is_noise(raw_text):
            item['reason'] = "Noise (No alphanumeric characters)"
            discarded_noise.append(item)
            continue

        processed_text = utils.default_process(raw_text)
        
        # Heuristic: For very short strings, we require a 100% match to avoid
        # false positives caused by common short phrases.
        current_threshold = 100 if len(raw_text) < 15 else threshold
        
        # Identify if the sample is already annotated
        entities = item.get("final_entities", []) or item.get("entities", [])
        has_labels = len(entities) > 0

        # Step 2: Fuzzy Matching
        match = process.extractOne(
            processed_text, 
            normalized_corpus, 
            scorer=fuzz.ratio,
            score_cutoff=current_threshold
        )
        
        if match:
            _, score, index = match
            item['similarity_score'] = score
            item['matched_with'] = corpus_as_list[index]
            
            if has_labels:
                item['reason'] = "Duplicate with labels"
                duplicates_with_labels.append(item)
            else:
                item['reason'] = "Duplicate (No labels)"
                duplicates_no_labels.append(item)
        else:
            # Step 3: Categorize Unique Samples
            if has_labels:
                item['reason'] = "Clean with labels"
                clean_labeled.append(item)
            else:
                item['reason'] = "Clean (No labels)"
                clean_unlabeled.append(item)
            
    # Print Summary Report
    print("\n" + "="*40)
    print("        FINAL DEDUPLICATION REPORT")
    print("="*40)
    print(f"  [GOLD] Clean Labeled:      {len(clean_labeled)}")
    print(f"  [INFO] Clean Unlabeled:    {len(clean_unlabeled)}")
    print("-" * 40)
    print(f"  [SKIP] Dups with Labels:   {len(duplicates_with_labels)}")
    print(f"  [SKIP] Dups No Labels:     {len(duplicates_no_labels)}")
    print(f"  [SKIP] Discarded Noise:    {len(discarded_noise)}")
    print("="*40)
    print(f"TOTAL PROCESSED: {len(new_candidates)}")
    
    return (
        clean_labeled, 
        clean_unlabeled, 
        duplicates_with_labels, 
        duplicates_no_labels, 
        discarded_noise
    )

In [ ]:
unique_list, dups_list = perform_internal_deduplication(
    candidates=train_candidates_data,
    threshold=95.0
)

In [ ]:
clean_lab, clean_unlab, dup_lab, dup_empty, noise = perform_deduplication(
    unique_list,
    all_existing_text,
    threshold=95.0
)

In [ ]:
for item in dup_lab[:10]:
    if item['similarity_score'] <= 98:
        print(item)

Now that we have successfully categorized our data into five distinct groups, we need to prepare the final training set. Our goal is to maintain a healthy ratio between **Annotated (Labeled)** and **Background (Unlabeled)** samples to prevent model bias.

### Current Data Inventory
We are working with the following sets:
1.  **Original Train**: Our existing training data.
2.  **Original Dev**: Validation set.
3.  **Original Test**: Evaluation set.
4.  **Clean Labeled (`clean_lab`)**: New high-value candidates with identified entities.
5.  **Clean Unlabeled (`clean_unlab`)**: New candidates containing only background text.

### The Balancing Logic
To optimize the Fine-Tuning process, we will:
* **Analyze the Original Train Set**: Count how many paragraphs contain at least one entity versus those that are purely background text.
* **Integrate New Data**: 
    * Add **all** samples from `clean_lab` to the training pool.
    * Sub-sample from `clean_unlab` so that the final number of unlabeled paragraphs matches the number of labeled paragraphs (or maintains the original ratio).

> **Goal:** Ensure that the "Unlabeled" data does not overwhelm the "Labeled" data, allowing the model to learn entity boundaries effectively without over-predicting the 'Outside' (O) tag.

---

### Implementation Step: Analyzing the Existing Train Set
In the next cell, we will parse the original `train.txt` to calculate the current ratio before merging the new candidates.

In [ ]:
def parse_paragraphs(lines: list) -> list:
    paragraphs = []
    current_tokens = []
    current_labels = []

    for raw_line in lines:
        line = raw_line.rstrip()
        if not line:
            if current_tokens:
                paragraph_text = " ".join(current_tokens).strip()
                paragraphs.append((paragraph_text, list(current_labels)))
                current_tokens = []
                current_labels = []
            continue

        try:
            token, label = line.rsplit(" ", 1)
        except ValueError:
            token = line
            label = ""

        current_tokens.append(token)
        current_labels.append(label.strip())

    if current_tokens:
        paragraph_text = " ".join(current_tokens).strip()
        paragraphs.append((paragraph_text, list(current_labels)))

    return paragraphs

In [ ]:
def calculate_label_distribution(parsed_paragraphs: list) -> dict:
    """
    Calculates the number of paragraphs that contain at least one entity
    versus those that contain only 'O' (background) tags.
    
    Args:
        parsed_paragraphs (list): List of tuples (text, list_of_labels)
        
    Returns:
        dict: A dictionary with counts and the specific lists of paragraphs.
    """
    labeled_samples = []
    unlabeled_samples = []

    for text, labels in parsed_paragraphs:
        # Check if any label in the list is NOT 'O' and NOT empty
        # We use a set for faster checking
        unique_labels = set(labels)
        
        # A paragraph is 'labeled' if it contains anything other than 'O' or ''
        has_entities = any(label not in ['O', ''] for label in unique_labels)

        if has_entities:
            labeled_samples.append((text, labels))
        else:
            unlabeled_samples.append((text, labels))

    total = len(parsed_paragraphs)
    stats = {
        "labeled_count": len(labeled_samples),
        "unlabeled_count": len(unlabeled_samples),
        "labeled_samples": labeled_samples,
        "unlabeled_samples": unlabeled_samples,
        "ratio": len(labeled_samples) / total if total > 0 else 0
    }

    print("--- Distribution Analysis ---")
    print(f"Total Paragraphs: {total}")
    print(f"With Entities:    {stats['labeled_count']} ({stats['ratio']:.1%})")
    print(f"Only Background:  {stats['unlabeled_count']} ({1 - stats['ratio']:.1%})")
    
    return stats

In [ ]:
train_lines = open(train_set_path, "r").readlines()
parsed_train = parse_paragraphs(train_lines)
train_stats = calculate_label_distribution(parsed_train)

In [ ]:
import random

def get_balanced_unlabeled_subset(
    original_train_stats: dict, 
    clean_lab_list: list, 
    clean_unlab_list: list, 
    seed: int = 42
) -> list:
    """
    Calculates the number of unlabeled samples needed to achieve a 50/50 ratio
    and selects them randomly from the clean_unlab pool.
    
    Args:
        original_train_stats: Dict from calculate_label_distribution (contains counts)
        clean_lab_list: List of new labeled candidates
        clean_unlab_list: List of new unlabeled candidates
        seed: Random seed for reproducibility
        
    Returns:
        list: The selected subset of clean_unlab candidates.
    """

    num_orig_lab = original_train_stats['labeled_count']
    num_new_lab = len(clean_lab_list)
    total_labeled_target = num_orig_lab + num_new_lab
    
    num_orig_unlab = original_train_stats['unlabeled_count']
    
    unlabeled_needed = total_labeled_target - num_orig_unlab
    
    print("--- Balancing Calculation ---")
    print(f"Target Labeled Total: {total_labeled_target}")
    print(f"Current Unlabeled in Train: {num_orig_unlab}")
    
    if unlabeled_needed <= 0:
        print(">> No new unlabeled candidates needed. The set is already balanced or heavy on background.")
        return []
    
    random.seed(seed)
    
    # Ensure we don't try to sample more than we have
    sample_size = min(unlabeled_needed, len(clean_unlab_list))
    
    selected_unlabeled = random.sample(clean_unlab_list, sample_size)
    
    print(f">> Required Unlabeled: {unlabeled_needed}")
    print(f">> Selected from clean_unlab: {len(selected_unlabeled)}")
    
    if len(selected_unlabeled) < unlabeled_needed:
        print(f"!! Warning: Could only find {len(selected_unlabeled)} samples, but needed {unlabeled_needed}.")
    
    return selected_unlabeled

In [ ]:
selected_unlab_candidates = get_balanced_unlabeled_subset(
    train_stats, 
    clean_lab, 
    clean_unlab, 
    seed=42
)

In [ ]:
selected_unlab_candidates[:1]

In [ ]:
import os

def export_clean_candidates(labeled_list, unlabeled_list, output_path):
    """
    Concatenates labeled and unlabeled candidates and saves them as a JSONL file.
    
    Args:
        labeled_list (list): The full list of clean_labeled candidates.
        unlabeled_list (list): The balanced subset of clean_unlabeled candidates.
        output_path (str): Full path (including filename.jsonl) where to save.
    """

    final_pool = labeled_list + unlabeled_list
    
    os.makedirs(os.path.dirname(output_path), exist_ok=True)
    
    try:
        with open(output_path, 'w', encoding='utf-8') as f:
            for entry in final_pool:
                json_record = json.dumps(entry, ensure_ascii=False)
                f.write(json_record + '\n')
        
        print("--- Export Success ---")
        print(f"File saved to: {output_path}")
        print(f"Total records exported: {len(final_pool)}")
        print(f"  - Labeled:   {len(labeled_list)}")
        print(f"  - Unlabeled: {len(unlabeled_list)}")
        
    except Exception as e:
        print(f"Error saving file: {e}")

In [ ]:
output_file_clean_candidates= "../../../resources/data/restricted/ner-review/new-data/merged_clean_candidates.jsonl"
export_clean_candidates(clean_lab, selected_unlab_candidates, output_file_clean_candidates)

In [ ]:
from collections import namedtuple, Counter

CharInterval = namedtuple('CharInterval', ['start_pos', 'end_pos'])

class Extraction:
    """Adapter class to match the interface expected by extractions_to_bio."""
    def __init__(self, extraction_class: str, start_pos: int, end_pos: int):
        self.extraction_class = extraction_class
        self.char_interval = CharInterval(start_pos, end_pos)

def build_token_offsets(text: str, tokens: List[str]) -> List[Tuple[int, int]]:
    """Calculates the start and end character positions for a list of tokens.

    Args:
        text: The raw source text.
        tokens: List of tokens (words with punctuation attached).

    Returns:
        A list of (start, end) character offsets.

    Raises:
        ValueError: If a token cannot be found in the text following the previous token.
    """
    offsets = []
    cursor = 0
    for token in tokens:
        start = text.find(token, cursor)
        if start == -1:
            raise ValueError(
                f"No pude alinear token '{token}' en '{text[cursor : cursor + 50]}'"
            )
        end = start + len(token)
        offsets.append((start, end))
        cursor = end
    return offsets

def extractions_to_bio(
    extractions: List[Any], 
    token_offsets: List[Tuple[int, int]], 
    default_label: str = "O"
) -> List[str]:
    """Maps entity extractions to token-level BIO labels based on char overlap.

    Args:
        extractions: List of objects with .char_interval and .extraction_class.
        token_offsets: Character offsets for each token.
        default_label: The tag to use for non-entity tokens.

    Returns:
        A list of BIO tags.
    """
    labels = [default_label] * len(token_offsets)
    for extraction in extractions:
        if not extraction.char_interval:
            continue
            
        cls = extraction.extraction_class
        start_char = extraction.char_interval.start_pos
        end_char = extraction.char_interval.end_pos
        first = True
        
        for i, (tok_start, tok_end) in enumerate(token_offsets):
            # Check if token range overlaps with entity range
            if end_char <= tok_start or start_char >= tok_end:
                continue
            
            prefix = "B-" if first else "I-"
            labels[i] = f"{prefix}{cls}"
            first = False
    return labels

def convert_json_to_bio(entry: Dict[str, Any]) -> List[str]:
    """Converts a JSON dictionary into BIO-tagged lines using split().

    Args:
        entry: Dictionary containing 'text' and 'entities'.

    Returns:
        List of "Token Label" strings. Empty list if alignment fails.
    """
    text = entry.get("text", "")
    entities_data = entry.get("entities", [])
    
    tokens = text.split()
    
    try:
        offsets = build_token_offsets(text, tokens)
    except ValueError as e:
        print(f"Skipping paragraph due to alignment error: {e}")
        return []

    extractions = [
        Extraction(ent['label'], ent['start_char'], ent['end_char']) 
        for ent in entities_data
    ]

    bio_tags = extractions_to_bio(extractions, offsets)
    return [f"{t} {l}" for t, l in zip(tokens, bio_tags)]

def finalize_training_set(
    original_path: str, 
    jsonl_path: str, 
    output_path: str, 
    overwrite: bool = False
):
    """Merges original BIO file with new JSONL data. 

    If output_path does not exist, it is created. If it exists, it prompts 
    for confirmation unless overwrite is True.

    Args:
        original_path: Path to existing train.txt.
        jsonl_path: Path to the JSONL with clean candidates.
        output_path: Path for the new consolidated BIO file.
        overwrite: If True, skips existence confirmation.
    """
    final_output_lines = []

    if os.path.exists(original_path):
        print(f"Loading existing data from {original_path}...")
        with open(original_path, 'r', encoding='utf-8') as f:
            final_output_lines = f.readlines()
            if final_output_lines and final_output_lines[-1].strip() != "":
                final_output_lines.append("\n")
    else:
        print(f"Original path {original_path} not found. Starting with a new dataset.")

    print(f"Processing new JSONL records from {jsonl_path}...")
    new_count = 0
    with open(jsonl_path, 'r', encoding='utf-8') as f:
        for line in tqdm(f, desc="Converting to BIO"):
            if not line.strip(): continue
            entry = json.loads(line)
            
            bio_formatted_lines = convert_json_to_bio(entry)
            if not bio_formatted_lines:
                continue
            
            final_output_lines.extend([f"{l}\n" for l in bio_formatted_lines])
            final_output_lines.append("\n") 
            new_count += 1

    if os.path.exists(output_path) and not overwrite:
        print(f"The file '{output_path}' already exists.")
        choice = input(f"Overwrite with {len(final_output_lines)} total lines? (y/n): ").lower()
        if choice != 'y':
            print("Aborted. No data was written.")
            return
    elif not os.path.exists(output_path):
        print(f"Output file does not exist. Creating '{output_path}'...")

    # 4. Write to Disk
    with open(output_path, 'w', encoding='utf-8') as f:
        f.writelines(final_output_lines)

    print(f"Merged {new_count} new paragraphs.")
    print(f"Final training set available at: {output_path}")

In [ ]:
output_path_final_train_set = "../../../resources/data/restricted/ner-review/new-data/train_expanded.txt"

finalize_training_set(
    original_path=train_set_path, 
    jsonl_path=output_file_clean_candidates, 
    output_path=output_path_final_train_set
)

In [ ]:
def create_balanced_bio_dataset(
    clean_labeled: List[Dict[str, Any]], 
    clean_unlabeled: List[Dict[str, Any]], 
    balance_ratio: float = 0.5, 
    output_path: str = "train_extended.txt",
    seed: int = 42,
    overwrite: bool = False
) -> str:
    """Creates a BIO-formatted text file by balancing labeled and unlabeled data.

    Combines annotated samples with a random subset of background samples. 
    Uses a fixed seed to ensure that the selection of unlabeled data is 
    consistent across different runs.

    Args:
        clean_labeled: List of dictionaries with 'text' and 'entities'.
        clean_unlabeled: List of dictionaries with 'text' and no entities.
        balance_ratio: The number of unlabeled samples to select relative 
            to the labeled count (e.g., 0.1 for 10% unlabeled).
        output_path: The file path where the .txt will be saved.
        seed: Integer used to initialize the random number generator for 
            reproducible results. Defaults to 42.
        overwrite: If True, skips the check for an existing file.

    Returns:
        The string path to the saved file if successful, otherwise an empty string.
    """

    if os.path.exists(output_path) and not overwrite:
        print(f"File '{output_path}' already exists.")
        confirm = input("Overwrite? (y/n): ").lower()
        if confirm != 'y':
            print("Operation aborted by user.")
            return ""

    random.seed(seed)

    num_labeled = len(clean_labeled)
    num_to_select = int(num_labeled * balance_ratio)
    
    num_to_select = min(num_to_select, len(clean_unlabeled))
    
    selected_unlabeled = random.sample(clean_unlabeled, num_to_select)
    
    final_pool = clean_labeled + selected_unlabeled
    random.shuffle(final_pool)

    print(f"Final Pool Composition: {num_labeled} Labeled + {num_to_select} Unlabeled")

    final_lines = []
    for entry in final_pool:
        bio_lines = convert_json_to_bio(entry)
        if not bio_lines:
            continue
            
        for line in bio_lines:
            final_lines.append(f"{line}\n")

        final_lines.append("\n")

    with open(output_path, 'w', encoding='utf-8') as f:
        f.writelines(final_lines)

    print(f"Successfully generated balanced corpus at: {output_path}")
    return output_path

In [ ]:
train_extended_path = "../../../resources/data/restricted/ner-flair-finetunning/train.txt"

In [ ]:
final_train_set = create_balanced_bio_dataset(
    clean_labeled=clean_lab,
    clean_unlabeled=clean_unlab,
    balance_ratio=1,
    seed=42,
    output_path=train_extended_path
)

## Dataset

In [ ]:
import flair
import torch
from flair.data import Corpus, Sentence
from flair.datasets import ColumnCorpus
from flair.embeddings import (
    FlairEmbeddings,
    StackedEmbeddings,
    TransformerWordEmbeddings,
)
from flair.models import SequenceTagger
from flair.tokenization import SpaceTokenizer
from flair.trainers import ModelTrainer
from flair.visual.training_curves import Plotter
from torch.optim.lr_scheduler import OneCycleLR

flair.device = torch.device("cuda")
torch.cuda.is_available()

In [ ]:
columns = {0: "text", 1: "ner"}

data_folder_old = "/Users/MacConra/Documents/Collective/AymurAI/backend/resources/data/restricted/ner-review/new-data"
data_folder_new = "/Users/MacConra/Documents/Collective/AymurAI/backend/resources/data/restricted/ner-flair-finetunning"

old_corpus = ColumnCorpus(
    data_folder_old,
    columns,
    train_file="train.txt",
    test_file="test_review.txt",
    dev_file="dev_review.txt",
)


new_corpus = ColumnCorpus(
    data_folder_new,
    columns,
    train_file="train.txt",
    test_file="test.txt",
    dev_file="dev.txt",
)

In [ ]:
label_type = "ner"

In [ ]:
len(new_corpus.train), len(new_corpus.dev), len(new_corpus.test)

In [ ]:
vocab_dictionary_new = new_corpus.make_vocab_dictionary()
vocab_dictionary_old = old_corpus.make_vocab_dictionary()

In [ ]:
label_dictionary_new = new_corpus.make_label_dictionary(label_type=label_type, add_unk=True)
label_dictionary_old = old_corpus.make_label_dictionary(label_type=label_type, add_unk=True)

In [ ]:
sentence = new_corpus.train[0]

print(sentence.to_plain_string())

for entity in sentence:
    # print(sentence.text[entity.start_position:entity.end_position])
    # print(new_corpus.train[0].text[entity.start_position:entity.end_position])
    print(entity.get_label('ner').value)
    print(entity.start_position, entity.end_position)
    print(entity.text)

In [ ]:
print(new_corpus.obtain_statistics())

print(old_corpus.obtain_statistics())

In [ ]:
from ast import literal_eval

import pandas as pd

stats_new = literal_eval(new_corpus.obtain_statistics())
stats_old = literal_eval(old_corpus.obtain_statistics())

## Comparative EDA: Old vs New Corpora

This section turns the next cells into a compact exploratory analysis of both corpus. The goal is to compare the **old** and **new** datasets across:
- split size and entity coverage
- relative label frequency
- sentence length and entity density
- lexical variety per label
- label consistency across train/dev/test


In [ ]:
from collections import Counter
from typing import Dict, List, Tuple

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from IPython.display import display
from matplotlib.ticker import PercentFormatter

sns.set_theme(style="whitegrid")

DATASETS = {
    "Old Data": {"corpus": old_corpus, "stats": stats_old, "color": "orange"},
    "New Data": {"corpus": new_corpus, "stats": stats_new, "color": "red"},
}
SPLITS = ["TRAIN", "DEV", "TEST"]


def get_split_sentences(corpus: Corpus, split_name: str) -> List[Sentence]:
    return getattr(corpus, split_name.lower())


def count_entities(sentence: Sentence, label_name: str = "ner") -> int:
    return len(sentence.get_spans(label_name))


def collect_sentence_level_stats(
    dataset: List[Sentence], label_name: str = "ner"
) -> pd.DataFrame:
    rows = []
    for idx, sentence in enumerate(dataset):
        entity_count = count_entities(sentence, label_name)
        token_count = len(sentence)
        rows.append(
            {
                "sentence_idx": idx,
                "token_count": token_count,
                "entity_count": entity_count,
                "entity_density": entity_count / token_count if token_count else 0,
                "has_entities": entity_count > 0,
            }
        )
    return pd.DataFrame(rows)


def build_split_summary() -> pd.DataFrame:
    rows = []
    for dataset_name, payload in DATASETS.items():
        corpus = payload["corpus"]
        stats = payload["stats"]
        for split_name in SPLITS:
            split_sentences = get_split_sentences(corpus, split_name)
            split_frame = collect_sentence_level_stats(split_sentences)
            rows.append(
                {
                    "dataset": dataset_name,
                    "split": split_name,
                    "sentences": len(split_sentences),
                    "tokens": int(split_frame["token_count"].sum()),
                    "entities": int(split_frame["entity_count"].sum()),
                    "sentences_with_entities_pct": round(split_frame["has_entities"].mean() * 100, 2),
                    "avg_tokens_per_sentence": round(split_frame["token_count"].mean(), 2),
                    "avg_entities_per_sentence": round(split_frame["entity_count"].mean(), 2),
                    "label_types": len(stats[split_name]["number_of_documents_per_class"]),
                }
            )
    return pd.DataFrame(rows)


def build_relative_label_distribution(split_name: str) -> pd.DataFrame:
    series_list = []
    for dataset_name, payload in DATASETS.items():
        label_counts = pd.Series(
            payload["stats"][split_name]["number_of_documents_per_class"],
            dtype=float,
        )
        label_freq_pct = (label_counts / label_counts.sum()) * 100
        series_list.append(label_freq_pct.rename(dataset_name))

    comparison = pd.concat(series_list, axis=1).fillna(0)
    comparison["abs_diff"] = (comparison["New Data"] - comparison["Old Data"]).abs()
    comparison = comparison.sort_values("abs_diff", ascending=False)
    return comparison


def plot_relative_label_distribution(split_name: str, top_n: int | None = None) -> pd.DataFrame:
    comparison = build_relative_label_distribution(split_name)
    plot_df = comparison.head(top_n) if top_n else comparison

    ax = plot_df[["Old Data", "New Data"]].plot(
        kind="bar",
        figsize=(14, 6),
        color=[DATASETS["Old Data"]["color"], DATASETS["New Data"]["color"]],
        width=0.8,
    )
    ax.set_title(f"{split_name.title()} Label Distribution Comparison (Relative %)")
    ax.set_ylabel("Percentage of Label Mentions")
    ax.set_xlabel("NER Labels")
    ax.yaxis.set_major_formatter(PercentFormatter())
    plt.xticks(rotation=70)
    plt.tight_layout()

    return comparison


def build_density_frame(label_name: str = "ner") -> pd.DataFrame:
    frames = []
    for dataset_name, payload in DATASETS.items():
        for split_name in SPLITS:
            frame = collect_sentence_level_stats(
                get_split_sentences(payload["corpus"], split_name),
                label_name=label_name,
            )
            frame["dataset"] = dataset_name
            frame["split"] = split_name
            frames.append(frame)
    return pd.concat(frames, ignore_index=True)


def analyze_lexical_variety(corpus_subset: List[Sentence], label_name: str = "ner") -> pd.DataFrame:
    variety = {}
    for sentence in corpus_subset:
        for span in sentence.get_spans(label_name):
            variety.setdefault(span.tag, []).append(span.text.lower())

    report = []
    for label, texts in variety.items():
        unique_texts = set(texts)
        report.append(
            {
                "Label": label,
                "Total Occurrences": len(texts),
                "Unique Variants": len(unique_texts),
                "Variety Ratio": round(len(unique_texts) / len(texts), 4) if texts else 0,
            }
        )
    return pd.DataFrame(report).sort_values(["Total Occurrences", "Variety Ratio"], ascending=[False, False])


def get_entity_context(
    corpus_subset: List[Sentence],
    target_label: str,
    label_name: str = "ner",
    window: int = 3,
    top_k: int = 10,
) -> List[Tuple[str, int]]:
    contexts = []
    for sentence in corpus_subset:
        tokens = [token.text for token in sentence]
        for span in sentence.get_spans(label_name):
            if span.tag != target_label:
                continue
            start = span.tokens[0].idx - 1
            end = span.tokens[-1].idx - 1
            pre = tokens[max(0, start - window):start]
            post = tokens[end + 1:end + 1 + window]
            contexts.append(f"{' '.join(pre)} [ENTITY] {' '.join(post)}".strip())
    return Counter(contexts).most_common(top_k)


def label_split_overlap_table(stats: Dict[str, Dict]) -> pd.DataFrame:
    split_labels = {
        split_name: set(stats[split_name]["number_of_documents_per_class"].keys())
        for split_name in SPLITS
    }
    all_labels = sorted(set().union(*split_labels.values()))
    rows = []
    for label in all_labels:
        row = {"label": label}
        for split_name in SPLITS:
            row[split_name] = label in split_labels[split_name]
        rows.append(row)
    return pd.DataFrame(rows)


In [ ]:
split_summary_df = build_split_summary()
display(split_summary_df[split_summary_df["split"] == "TRAIN"])

In [ ]:
train_label_comparison_df = plot_relative_label_distribution("TRAIN")
display(train_label_comparison_df.round(2))


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for dataset_name, payload in DATASETS.items():
    for split_name in SPLITS:
        label_counts = pd.Series(
            payload["stats"][split_name]["number_of_documents_per_class"],
            dtype=float,
        )
        if label_counts.empty:
            continue
        label_freq_pct = (label_counts / label_counts.sum()) * 100
        axes[0].plot(
            label_freq_pct.sort_values(ascending=False).reset_index(drop=True),
            marker="o",
            label=f"{dataset_name} - {split_name}",
            alpha=0.8,
        )

summary_slice = split_summary_df.copy()
summary_slice["split_dataset"] = summary_slice["dataset"] + " - " + summary_slice["split"]
axes[1].bar(
    summary_slice["split_dataset"],
    summary_slice["sentences_with_entities_pct"],
    color=[DATASETS[row.dataset]["color"] for row in summary_slice.itertuples()],
)

axes[0].set_title("Label Concentration Curves by Split")
axes[0].set_ylabel("Relative Frequency (%)")
axes[0].set_xlabel("Labels Ranked by Frequency")
axes[0].yaxis.set_major_formatter(PercentFormatter())
axes[0].legend(fontsize=9)

axes[1].set_title("Sentences with Entities by Split")
axes[1].set_ylabel("Percentage of Sentences")
axes[1].yaxis.set_major_formatter(PercentFormatter())
axes[1].tick_params(axis="x", rotation=45)

plt.tight_layout()


In [ ]:
density_df = build_density_frame()

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

sns.boxplot(
    data=density_df[density_df["split"] == "TRAIN"],
    y="token_count",
    hue="dataset",
    ax=axes[0],
)
axes[0].set_title("Sentence Length Distribution")
axes[0].set_xlabel("Train Split")
axes[0].set_ylabel("Tokens per Sentence")

sns.boxplot(
    data=density_df[density_df["split"] == "TRAIN"],
    y="entity_density",
    hue="dataset",
    ax=axes[1],
)
axes[1].set_title("Entity Density Distribution")
axes[1].set_xlabel("Train Split")
axes[1].set_ylabel("Entities / Token")

for ax in axes:
    ax.legend(title="Dataset")

plt.tight_layout()

display(
    density_df[density_df["split"] == "TRAIN"].groupby(["dataset"])[["token_count", "entity_count", "entity_density"]]
    .agg(["mean", "median", "std"])
    .round(3)
)


In [ ]:
coverage_tables = {}
for dataset_name, payload in DATASETS.items():
    coverage_table = label_split_overlap_table(payload["stats"])
    coverage_tables[dataset_name] = coverage_table
    print(f"\n=== {dataset_name} label coverage across splits ===")
    display(coverage_table)


In [ ]:
old_train_variety = analyze_lexical_variety(old_corpus.train)
old_train_variety["Dataset"] = "Old Data"

new_train_variety = analyze_lexical_variety(new_corpus.train)
new_train_variety["Dataset"] = "New Data"

lexical_variety_comparison = pd.concat([old_train_variety, new_train_variety], ignore_index=True)
display(lexical_variety_comparison.sort_values(["Label", "Dataset"]).reset_index(drop=True))


In [ ]:
shared_train_labels = sorted(
    set(stats_old["TRAIN"]["number_of_documents_per_class"].keys())
    & set(stats_new["TRAIN"]["number_of_documents_per_class"].keys())
)

label_context_report = {}
for label in shared_train_labels[: min(5, len(shared_train_labels))]:
    label_context_report[label] = {
        "Old Data": get_entity_context(old_corpus.train, label, top_k=5),
        "New Data": get_entity_context(new_corpus.train, label, top_k=5),
    }

label_context_report


In [ ]:
label_shift_df = train_label_comparison_df.copy()
label_shift_df["new_minus_old"] = label_shift_df["New Data"] - label_shift_df["Old Data"]
display(label_shift_df.sort_values("new_minus_old", ascending=False).round(2))


In [ ]:
split_summary_df.sort_values(["dataset", "split"]).reset_index(drop=True)


In [ ]:
from pathlib import Path

old_label_counts = pd.Series(
    stats_old["TRAIN"]["number_of_documents_per_class"],
    dtype=float,
)

old_label_frequency_pct = ((old_label_counts / old_label_counts.sum()) * 100).rename(
    "relative_frequency_pct"
)

least_frequent_old_labels_top15 = (
    old_label_frequency_pct.sort_values(ascending=True)
    .head(15)
    .reset_index()
    .rename(columns={"index": "label"})
)

# Tab-separated table: easy to load with pandas.read_csv(..., sep='\t') from another notebook.
top15_output_txt_path = Path("top-15-less-frequent-labels.txt")
least_frequent_old_labels_top15.to_csv(
    top15_output_txt_path,
    sep="\t",
    index=False,
)

# Plain one-label-per-line file: easy to load and reuse as a Python list.
unique_train_labels = sorted(stats_old["TRAIN"]["number_of_documents_per_class"].keys())
unique_labels_output_txt_path = Path("original-train-unique-labels.txt")
unique_labels_output_txt_path.write_text(
    "\n".join(unique_train_labels),
    encoding="utf-8",
)

print(f"Saved top-15 table to: {top15_output_txt_path.resolve()}")
print(f"Saved unique label list to: {unique_labels_output_txt_path.resolve()}")
least_frequent_old_labels_top15
